In [ ]:
### gradient off displacement maps

# steps

# 0: load dependencies
# 1: Define functions
# 2: load data
# 3: define grid spacing and kernel width
# 4: calculate gradient in range and azimuth, get direction of maximum gradient
# 7: plot results


In [ ]:
# load dependencies

## first time series test
# append ../../ to make link to main repo work
import sys
sys.path.append('../../')
sys.path.append('/Applications/anaconda3/envs/PhD_clean/lib/python3.11/site-packages/')


# import external packages
import numpy as np
import pandas as pd
import numba
from numba import vectorize
import glob # for file search
import copy
import os # operating system stuff
import re # regex


import geopandas as gpd
import rasterio as rio
import rasterio.mask
from rasterio.plot import plotting_extent
# import rioxarray
from shapely.geometry import Polygon
from shapely.geometry.point import Point
import pyproj
from pyproj import CRS
from inpoly import inpoly2 # for fast inpolygon checks
import utm

import matplotlib.pyplot as plt 
import matplotlib.dates as mdates
from matplotlib import cm as mpl_cm
from matplotlib import colors as mcolors 
import matplotlib.image as mplimg

from mpl_toolkits.axes_grid1 import make_axes_locatable # for colorbar scaling
from mpl_toolkits.axes_grid1 import ImageGrid
from matplotlib_scalebar.scalebar import ScaleBar
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FormatStrFormatter

import seaborn as sns
from matplotlib import rc_file_defaults
rc_file_defaults()
# sns.set(style=None, color_codes=True)

from shapely.geometry import Polygon
from shapely.geometry.point import Point
import datetime

import configparser

from cmcrameri import cm # for scientific colourmaps

from scipy import spatial
# from scikit_spatial import Line as sksline
# import scipy.spatial.transform 

from skspatial.objects import Line as sksline
import scipy.spatial.transform 

import json
from shapely.geometry import Point, LineString, mapping
from functools import partial
from shapely.ops import transform

import sys
sys.executable


###########################
# import main local package
import SPOTSAR_main as sm



In [ ]:
### load data
from SPOTSAR_main.misc_func.read_line_shp import read_line_shp
# load SPOT time-series data
# CSK
CSK_TS_FILE = '/Users/tz20896/Documents/PhD/projects/Merapi2021/CSK/dsc1/CSK_SPOT_dsc_win13_ts_inv.csv'
CSK_TS = pd.read_csv(CSK_TS_FILE)
# TSX
TSX_TS_FILE = '/Users/tz20896/Documents/PhD/projects/Merapi2021/TSX/staring_134/TSX_ST134_win13_ts_inv.csv'
TSX_TS = pd.read_csv(TSX_TS_FILE)

# show TS data

print(CSK_TS)
print(TSX_TS)

col_offset = 3
CSK_R_disp_1 = (CSK_TS.iloc[:,col_offset+5 *2] - CSK_TS.iloc[:,col_offset+0*2])
CSK_R_disp_2 = (CSK_TS.iloc[:,col_offset+13*2] - CSK_TS.iloc[:,col_offset +5*2])
CSK_R_disp_3 = (CSK_TS.iloc[:,col_offset+37*2] - CSK_TS.iloc[:,col_offset+13*2])
CSK_R_disp_4 = (CSK_TS.iloc[:,col_offset+42*2] - CSK_TS.iloc[:,col_offset+37*2])
CSK_R_disp_5 = (CSK_TS.iloc[:,col_offset+54*2] - CSK_TS.iloc[:,col_offset+42*2])

CSK_A_disp_1 = (CSK_TS.iloc[:,col_offset+5 *2+1] - CSK_TS.iloc[:,col_offset+0*2+1])
CSK_A_disp_2 = (CSK_TS.iloc[:,col_offset+13*2+1] - CSK_TS.iloc[:,col_offset +5*2+1])
CSK_A_disp_3 = (CSK_TS.iloc[:,col_offset+37*2+1] - CSK_TS.iloc[:,col_offset+13*2+1])
CSK_A_disp_4 = (CSK_TS.iloc[:,col_offset+42*2+1] - CSK_TS.iloc[:,col_offset+37*2+1])
CSK_A_disp_5 = (CSK_TS.iloc[:,col_offset+54*2+1] - CSK_TS.iloc[:,col_offset+42*2+1])

CSK_Mag_1 = np.hypot(CSK_A_disp_1,CSK_R_disp_1)
CSK_Mag_2 = np.hypot(CSK_A_disp_2,CSK_R_disp_2)
CSK_Mag_3 = np.hypot(CSK_A_disp_3,CSK_R_disp_3)
CSK_Mag_4 = np.hypot(CSK_A_disp_4,CSK_R_disp_4)
CSK_Mag_5 = np.hypot(CSK_A_disp_5,CSK_R_disp_5)



TSX_R_disp_2 = (TSX_TS.iloc[:,col_offset+4*2] - TSX_TS.iloc[:,col_offset +0*2])
TSX_R_disp_3 = (TSX_TS.iloc[:,col_offset+12*2] - TSX_TS.iloc[:,col_offset+4*2])
TSX_R_disp_4 = (TSX_TS.iloc[:,col_offset+13*2] - TSX_TS.iloc[:,col_offset+12*2])
TSX_R_disp_5 = (TSX_TS.iloc[:,col_offset+14*2] - TSX_TS.iloc[:,col_offset+13*2])


TSX_A_disp_2 = (TSX_TS.iloc[:,col_offset+4*2+1] - TSX_TS.iloc[:,col_offset +0*2+1])
TSX_A_disp_3 = (TSX_TS.iloc[:,col_offset+12*2+1] - TSX_TS.iloc[:,col_offset+4*2+1])
TSX_A_disp_4 = (TSX_TS.iloc[:,col_offset+13*2+1] - TSX_TS.iloc[:,col_offset+12*2+1])
TSX_A_disp_5 = (TSX_TS.iloc[:,col_offset+14*2+1] - TSX_TS.iloc[:,col_offset+13*2+1])


TSX_Mag_2 = np.hypot(TSX_A_disp_2,TSX_R_disp_2)
TSX_Mag_3 = np.hypot(TSX_A_disp_3,TSX_R_disp_3)
TSX_Mag_4 = np.hypot(TSX_A_disp_4,TSX_R_disp_4)
TSX_Mag_5 = np.hypot(TSX_A_disp_5,TSX_R_disp_5)

# DEM
Merapi_DEM_file = "/Users/tz20896/Documents/PhD/projects/Merapi2021/geo/TDX_Merapi_WGS84_5m.tif"
with rio.open(Merapi_DEM_file) as img:
    DEM = img.read(1,masked=True) # rasterio bands are indexed from 1
    # extract DEM extent
    DEM_extent=[img.bounds.left,img.bounds.right,img.bounds.bottom,img.bounds.top]
    print('Band1 has shape', DEM.shape)
    height = DEM.shape[0]
    width  = DEM.shape[1]
    cols, rows = np.meshgrid(np.arange(width), np.arange(height))
    xs, ys = rio.transform.xy(img.transform, rows, cols)
    lons_DEM = np.array(xs)
    lats_DEM = np.array(ys)


# DEM
Merapi_HS_file = "/Users/tz20896/Documents/PhD/projects/Merapi2021/geo/TDX_Merapi_WGS84_HS.tif"
with rio.open(Merapi_HS_file) as img:
    SHADING = img.read(1,masked=True) # rasterio bands are indexed from 1


### lava flows and crater
L1888_FILE = ("/Users/tz20896/Documents/PhD/projects/Merapi2021/merapi_maps/L1888_v2.shp")
L1948_FILE = ("/Users/tz20896/Documents/PhD/projects/Merapi2021/merapi_maps/L1948.shp")
L1956_FILE = ("/Users/tz20896/Documents/PhD/projects/Merapi2021/merapi_maps/L1956.shp")
L1992_FILE = ("/Users/tz20896/Documents/PhD/projects/Merapi2021/merapi_maps/L1992.shp")
L1997_FILE = ("/Users/tz20896/Documents/PhD/projects/Merapi2021/merapi_maps/L1997.shp")
L1998_FILE = ("/Users/tz20896/Documents/PhD/projects/Merapi2021/merapi_maps/L1998.shp")
L1986_FILE = ("/Users/tz20896/Documents/PhD/projects/Merapi2021/merapi_maps/L1986.shp")
L1984_FILE = ("/Users/tz20896/Documents/PhD/projects/Merapi2021/merapi_maps/L1984.shp")
L1931_FILE = ("/Users/tz20896/Documents/PhD/projects/Merapi2021/merapi_maps/L1931.shp")

OLD_CRATER_FILE = ("/Users/tz20896/Documents/PhD/projects/Merapi2021/merapi_maps/old_crater_rims.shp")
CRATER_FILE = "/Users/tz20896/Documents/PhD/projects/Merapi2021/merapi_maps/Merapi_crater.shp"


# Extract latitude and longitude into separate columns
coords_L1888       = read_line_shp(L1888_FILE )
coords_L1931       = read_line_shp(L1931_FILE )
coords_L1948       = read_line_shp(L1948_FILE )
coords_L1956       = read_line_shp(L1956_FILE )
coords_L1984       = read_line_shp(L1984_FILE )
coords_L1986       = read_line_shp(L1986_FILE )
coords_L1992       = read_line_shp(L1992_FILE )
coords_L1997       = read_line_shp(L1997_FILE )
coords_L1998       = read_line_shp(L1998_FILE )
coords_CRATER      = read_line_shp(CRATER_FILE)
coords_OLD_CRATER  = read_line_shp(OLD_CRATER_FILE)


# separate multi-line
coords_L1956_1 = coords_L1956[0][0]
coords_L1956_2 = coords_L1956[1][0]

coords_pas1_CRATER   = coords_OLD_CRATER[0][0]
coords_pas2_CRATER   = coords_OLD_CRATER[1][0]
coords_1822_CRATER   = coords_OLD_CRATER[2][0]
coords_1872_CRATER   = coords_OLD_CRATER[3][0]
coords_1904_1_CRATER = coords_OLD_CRATER[4][0]
coords_1904_2_CRATER = coords_OLD_CRATER[5][0]
coords_1932_CRATER   = coords_OLD_CRATER[6][0]
coords_1961_1_CRATER = coords_OLD_CRATER[7][0]
coords_1961_2_CRATER = coords_OLD_CRATER[8][0]

In [ ]:
def calc_2Dgradient(Lon,Lat,R_off,A_off,grid,kernel_size):
    """
    Calculates range and azimuth displacement gradients for a field of scatter points given a defined grid and a kernel size.
    Inputs:
        Lon (np.array): array of longitude values
        Lat (np.arraY): array of latitude values
        R_off (np.array): array of range offset values
        A_off (np.array): array of azimuth offset values
        grid (np.array n*m*2): array of lon/lat coordinates to center the kernel on
        kernel size (tuple size=2): sizer of kernel in east and north directions.  
    """

    
